# Regressione logistica

è un modello di classificazione che predice la probabilità di appartenenza a una determinata classe.
La regressione logistica è un modello semplice, veloce, interpretabile e molto usato come baseline per problemi di classificazione binaria o multiclasse

In [1]:
import pandas as pd
import numpy as np


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.multioutput import MultiOutputClassifier




from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [ ]:
def training(file_path, csv_name):
    df = pd.read_csv(file_path)
    
    # Mi definisco la lista dei target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Filtro solo le pazienti con PR valido
    df_validi = df.dropna(subset=original_target_list).copy()

    # Binarizzazione forzata dei target per robustezza e per evitare errori
    # Questa operazione garantisce che i target siano sempre 0 o 1, indipendentemente
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  # Soglia clinica comune per KI67
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """ 
        Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempie eventuali valori mancanti (NaN) rimasti nelle colonne delle feature
    # Evito errori durante il training
    features = features.fillna(features.mean())


    # Istanzio il classificatore base
    base_clf = LogisticRegression(
        max_iter=50000,     # numero massimo di iterazioni
        random_state=42,    
        solver='saga',      # specifico l'algoritmo da usare, in questo caso "saga"
        tol=1e-3            # tolleranza per il criterio di arresto
    )
    
    # Lo avvolgo con MultiOutputClassifier per gestire i 3 target
    rf = MultiOutputClassifier(base_clf)

    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5)


    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # Itero manualmente attraverso le 5 fold definite da GroupKFold.
    # 'enumerate' tiene traccia del numero della fold corrente.
    for fold, (train_index, test_index) in enumerate(cv.split(features, target, groups)):
        
        # Suddivide i dati in set di training e di test per la fold corrente.
        X_train, X_test = features.iloc[train_index], features.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]

        # ========== DEBUGGING: Stampo indici train/test  ==========

         
        print("?"*50 + "\nDebug\n" + "?"*50)
        print(f"\nFold {fold} - File: {csv_name}")        
        print(f"  Train indice: {train_index[:10]})")
        print(f"  Test indice: {test_index[:10]})")
        print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
        print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
        print("?"*100)
        
        
        # ==========================================================


        # Verifica se nel set di test, per ogni target, sono presenti entrambe le classi (0 e 1).
        is_fold_valid = all(y_test[col].nunique() >= 2 for col in y_test.columns)
        
        if not is_fold_valid:
            # Se una fold contiene solo classi positive per un target,
            # il calcolo dell'F1-score fallirebbe. Quindi assegno uno score di 0 (il peggiore)
            # e saltiamo al prossimo ciclo per evitare errori.
            print(f"ATTENZIONE: Fold {fold} del file {csv_name} è invalida e viene assegnato score 0.")
            scores.append(0.0)
            continue

        # Qui presumo che la fold sia valida, quindi inizio il training
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        
        # Calcola l'F1-score
        score = f1_score(y_test, y_pred, average='micro', zero_division=0)
        scores.append(score)

    # Converto la lista di punteggi in un array numpy per facilitare i calcoli.
    scores = np.array(scores)

    return {
        'mean_score': scores.mean(),      # Performance media sulle 5 fold
        'std_score': scores.std(),        # Variabilità della performance
        'scores_per_fold': scores         # Lista dei 5 punteggi individuali
    }


# Lettura dei file

In [3]:
results = {}
print("="*50 + "\n Regressione Logistica\n" + "="*50)
for name, file_path in datasets.items():
    results[name] = training(file_path, name)

# Stampo i risultati 
for name, metrics in results.items():
    # Estraggo i 5 punteggi per il modello corrente
    scores_per_fold = metrics['scores_per_fold']
    
    # Formatto i punteggi in una stringa pulita
    formatted_scores = [f'{s:.3f}' for s in scores_per_fold]
    
    # Stampo la riga per il modello corrente
    print(f"\nNome CSV: {name}")
    print(f"    scores per forld: {formatted_scores}")
    print(f"    Media e Dev. Std.: {metrics['mean_score']:.3f} ± {metrics['std_score']:.3f}")

 Regressione Logistica
??????????????????????????????????????????????????
Debug
??????????????????????????????????????????????????

Fold 0 - File: t2_medsam
  Train indice: [ 0  2  3  4  5  7  8  9 10 11]... (totale: 47)
  Test indice: [ 1  6 18 24 28 33 38 43 44 45]... (totale: 12)
  Train gruppo (Patient IDs): ['AMBL-005' 'AMBL-008' 'AMBL-011' 'AMBL-016' 'AMBL-018' 'AMBL-028'
 'AMBL-029' 'AMBL-031' 'AMBL-038' 'AMBL-050' 'AMBL-496' 'AMBL-507'
 'AMBL-541' 'AMBL-557' 'AMBL-564' 'AMBL-568' 'AMBL-569' 'AMBL-570'
 'AMBL-571' 'AMBL-577' 'AMBL-580' 'AMBL-582' 'AMBL-583' 'AMBL-585'
 'AMBL-586' 'AMBL-591' 'AMBL-593' 'AMBL-596' 'AMBL-597' 'AMBL-598'
 'AMBL-599' 'AMBL-604' 'AMBL-605' 'AMBL-607' 'AMBL-608' 'AMBL-613'
 'AMBL-618' 'AMBL-619' 'AMBL-622' 'AMBL-628' 'AMBL-572' 'AMBL-574'
 'AMBL-579' 'AMBL-590' 'AMBL-631']
  Test gruppo (Patient IDs): ['AMBL-007' 'AMBL-022' 'AMBL-565' 'AMBL-578' 'AMBL-584' 'AMBL-594'
 'AMBL-603' 'AMBL-610' 'AMBL-612' 'AMBL-625']
????????????????????????????????????????